# M2 Homework — Vector Search
#### Juan Belbey

## Q1 — Embedding a query

Loading the ONNX-based Embedder (no PyTorch needed) and encode a query.
The model returns a normalized vector of 384 numbers. The answer is v[0].

In [1]:
# add current dir to path so Python can find embedder.py
import sys
sys.path.insert(0, '.')

from embedder import Embedder

# same model as sentence-transformers but ~33x lighter, no PyTorch needed
embed = Embedder()

# query from Q1
query = "How does approximate nearest neighbor search work?"

# encode() returns a normalized vector of 384 numbers
v = embed.encode(query)

print(f"Vector shape: {v.shape}")
print(f"v[0] = {v[0]:.4f}")  # Q1 answer

2026-06-26 13:14:18.715729582 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Vector shape: (384,)
v[0] = -0.0206


## Loading the data

Pull 72 lesson pages from the course repo, pinned to commit `8c1834d` so everyone works with the same data.

In [2]:
from gitsource import GithubRepositoryDataReader

# pinned to commit 8c1834d so everyone works with the same 72 pages
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

print(f"Total documents: {len(documents)}")
print(f"Example keys: {list(documents[0].keys())}")

Total documents: 72
Example keys: ['content', 'filename']


In [3]:
# find the specific doc from Q2
target = "02-vector-search/lessons/07-sqlitesearch-vector.md"
doc = next(d for d in documents if d["filename"] == target)

print(f"Found: {doc['filename']}")
# embed the doc content
doc_vector = embed.encode(doc["content"])

Found: 02-vector-search/lessons/07-sqlitesearch-vector.md


## Q2 — Cosine similarity

Embed the content of a specific doc and compute cosine similarity with the Q1 query vector.
Since both vectors are normalized, dot product = cosine similarity.

In [4]:
# dot product = cosine similarity (both vectors are normalized)
similarity = v.dot(doc_vector)
print(f"Cosine similarity: {similarity:.4f}")

Cosine similarity: 0.3611


## Q3 — Chunking and search by hand

Chunk all documents, embed every chunk into matrix X, score against Q1 query vector with a dot product, and find the top chunk.

In [5]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")
print(f"Chunk keys: {list(chunks[0].keys())}")

Total chunks: 295
Chunk keys: ['start', 'content', 'filename']


In [6]:
import numpy as np

# embed all chunks at once — returns a matrix
contents = [chunk["content"] for chunk in chunks]
X = np.array(embed.encode_batch(contents))

# score every chunk against the Q1 query vector
scores = X.dot(v)

# find the highest scoring chunk
best_idx = np.argmax(scores)
print(f"Best score: {scores[best_idx]:.4f}")
print(f"Filename: {chunks[best_idx]['filename']}")

Best score: 0.6489
Filename: 02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4 — Vector search with minsearch

Index the chunks with `minsearch.VectorSearch`, search with a new query, and return the filename of the first result.

In [7]:
from minsearch import VectorSearch

# build the index: vectors from Q3 (X) paired with their chunks
vs_index = VectorSearch()
vs_index.fit(X, chunks)

# new query for Q4 — different topic than Q1
q4_query = "What metric do we use to evaluate a search engine?"
q4_vector = embed.encode(q4_query)

# search returns the top chunks as dicts directly
results = vs_index.search(q4_vector, num_results=5)

# Q4 answer: filename of the first result
for i, r in enumerate(results):
    print(f"{i+1}. {r['filename']}")

1. 04-evaluation/lessons/05-search-metrics.md
2. 04-evaluation/lessons/01-intro.md
3. 01-agentic-rag/lessons/05-search.md
4. 04-evaluation/lessons/01-intro.md
5. 04-evaluation/lessons/15-next-steps.md


## Q5 — Text search vs vector search

Index the same chunks with `minsearch.Index` (keyword search). Run both searches for the same query and find which file appears in vector results but not in text results.

In [8]:
from minsearch import Index

# build keyword index with content as text field
text_index = Index(text_fields=["content"], keyword_fields=[])
text_index.fit(chunks)

# same query for both searches
q5_query = "How do I store vectors in PostgreSQL?"
q5_vector = embed.encode(q5_query)

# vector search (reuse vs_index from Q4)
vector_results = vs_index.search(q5_vector, num_results=5)

# keyword search
text_results = text_index.search(q5_query, num_results=5)

# compare filenames
vector_files = {r["filename"] for r in vector_results}
text_files = {r["filename"] for r in text_results}

print("Vector results:")
for r in vector_results:
    print(f"  {r['filename']}")

print("\nText results:")
for r in text_results:
    print(f"  {r['filename']}")

print("\nIn vector but NOT in text:")
for f in vector_files - text_files:
    print(f"  {f}")

Vector results:
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md

Text results:
  02-vector-search/lessons/02-embeddings.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md

In vector but NOT in text:
  02-vector-search/lessons/08-pgvector.md


## Q6 — Hybrid search con RRF

Combinamos vector search y text search con Reciprocal Rank Fusion.
RRF ignora los scores crudos y usa solo la posición de cada doc en cada lista.
Un doc que aparece bien rankeado en ambas listas acumula más score que uno fuerte en una sola.

In [9]:
q6_query = "How do I give the model access to tools?"
q6_vector = embed.encode(q6_query)

vector_results_q6 = vs_index.search(q6_vector, num_results=5)
text_results_q6 = text_index.search(q6_query, num_results=5)

print("Vector search ranking:")
for i, r in enumerate(vector_results_q6):
    print(f"  {i+1}. {r['filename']}")

print("\nText search ranking:")
for i, r in enumerate(text_results_q6):
    print(f"  {i+1}. {r['filename']}")

Vector search ranking:
  1. 01-agentic-rag/lessons/01-intro.md
  2. 04-evaluation/lessons/02-ground-truth.md
  3. 01-agentic-rag/lessons/16-other-frameworks.md
  4. 01-agentic-rag/lessons/15-frameworks.md
  5. 01-agentic-rag/lessons/13-function-calling.md

Text search ranking:
  1. 01-agentic-rag/lessons/14-agentic-loop.md
  2. 01-agentic-rag/lessons/13-function-calling.md
  3. 01-agentic-rag/lessons/13-function-calling.md
  4. 01-agentic-rag/lessons/13-function-calling.md
  5. 04-evaluation/lessons/02-ground-truth.md


In [10]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


hybrid_results = rrf([vector_results_q6, text_results_q6])

print("Hybrid search (RRF):")
for i, r in enumerate(hybrid_results):
    print(f"  {i+1}. {r['filename']}")

Hybrid search (RRF):
  1. 01-agentic-rag/lessons/13-function-calling.md
  2. 01-agentic-rag/lessons/01-intro.md
  3. 01-agentic-rag/lessons/14-agentic-loop.md
  4. 04-evaluation/lessons/02-ground-truth.md
  5. 01-agentic-rag/lessons/16-other-frameworks.md
